# Notebook 5 — Candidate Prioritization, Charger Sizing, and Final Output Tables


## Why this notebook exists
Notebook 4 transformed the mobility-driven candidate layer into a preliminary grid-feasibility screen by matching each candidate site to its nearest distributor node and assigning a first `grid_status`.

The role of this notebook is to reduce that candidate universe into a smaller proposal, assign a preliminary number of chargers per selected site, and reshape the results into the required datathon output tables.

In [1]:
import os
import pandas as pd
import geopandas as gpd

print("Files in current session:")
for f in os.listdir("/content"):
    print("-", f)

if os.path.exists("/content/candidate_points_grid_matched_v1.csv"):
    candidates = pd.read_csv("/content/candidate_points_grid_matched_v1.csv")
    print("\nLoaded CSV")
elif os.path.exists("/content/candidate_points_grid_matched_v1.gpkg"):
    candidates = gpd.read_file("/content/candidate_points_grid_matched_v1.gpkg")
    print("\nLoaded GPKG")
else:
    raise FileNotFoundError("Upload candidate_points_grid_matched_v1.csv or .gpkg into the Colab sidebar first.")

print("\nShape:", candidates.shape)
print("Columns:")
print(list(candidates.columns))

display(candidates.head(10))

Files in current session:
- .config
- candidate_points_grid_matched_v2.gpkg
- .ipynb_checkpoints
- candidate_points_grid_matched_v1.csv
- sample_data

Loaded CSV

Shape: (57, 28)
Columns:
['candidate_id_n4', 'id_catalog', 'carretera', 'clase', 'provincia', 'dist_to_backbone_m', 'distributor_network', 'province', 'municipality', 'substation_name', 'available_capacity_mw', 'occupied_capacity_mw', 'voltage_levels', 'n_raw_rows', 'source_substation_names', 'comments', 'dist_m', 'dist_to_grid_km', 'grid_status', 'n_nodes_in_radius', 'n_sufficient_in_radius', 'n_moderate_in_radius', 'best_capacity_in_radius', 'nearest_node_km', 'grid_match_flag', 'used_alternative_node', 'latitude', 'longitude']


,candidate_id_n4,id_catalog,carretera,clase,provincia,dist_to_backbone_m,distributor_network,province,municipality,substation_name,...,grid_status,n_nodes_in_radius,n_sufficient_in_radius,n_moderate_in_radius,best_capacity_in_radius,nearest_node_km,grid_match_flag,used_alternative_node,latitude,longitude
0,1,5545404112D766B32749F9697505489A,A-66,Autovía,Cáceres,5.990023e-11,i-DE,CÃ¡ceres,Malpartida de Plasencia,PANTANO GARGUERA,...,Moderate,10,0,1,6.0,8.148174,Far from nearest node,True,39.951364,-6.186186
1,2,5545404118556C932749F96974051277,A-66,Autovía,León,7.478352e-12,i-DE,LeÃ³n,VillamaÃ±Ã¡n,VILLAMAÃAN,...,Congested,21,0,0,0.0,7.636217,Medium distance to node,False,42.385657,-5.595729
2,3,554540411875EEB92749F96975054D28,A-66,Autovía,Cáceres,5.327118e-11,i-DE,CÃ¡ceres,Casar de CÃ¡ceres,CASAR CACERES,...,Congested,2,0,0,0.0,22.649347,Far from nearest node,False,39.755251,-6.399656
3,4,5545404118DF66332749F96975055401,A-66,Autovía,Cáceres,1.524195e-10,i-DE,CÃ¡ceres,CÃ¡ceres,CACERES2,...,Congested,11,0,0,0.0,13.336674,Medium distance to node,False,39.341053,-6.340569
4,5,5545404138D7CE192749F9697405168F,A-66,Autovía,Zamora,2.912580e-10,i-DE,Zamora,Benavente,VENTOSA,...,Congested,8,0,0,0.0,6.283722,Medium distance to node,False,41.944338,-5.634713
5,6,55454041907FEEB92749F96974052772,A-66,Autovía,Salamanca,1.161209e-11,i-DE,Salamanca,Castellanos de Villiquera,CASTELLANOS,...,Congested,14,0,0,0.0,12.939568,Medium distance to node,False,41.179327,-5.695357
6,7,55454049B2F56E842749F969740525E3,A-66,Autovía,Zamora,1.628323e-11,i-DE,Zamora,Valcabado,ZAMORA,...,Sufficient,10,1,2,32.1,3.087142,Far from nearest node,True,41.339153,-5.722461
7,8,5545404138DDE4112749F969B81A12F8,A-7,Autovía,Granada,8.017892e-11,Endesa,Granada,Gualchos,GUALCHOS,...,Congested,5,0,0,0.0,12.258611,Medium distance to node,False,36.757091,-3.241980
8,9,554540413AD7EC3B2749F969B81A14F6,A-7,Autovía,Granada,1.186558e-10,Endesa,Granada,Gualchos,CONJURO,...,Congested,9,0,0,0.0,3.206971,Near node,False,36.711086,-3.415317
9,10,5545404192F5E4912749F969B81A1778,A-7,Autovía,Granada,1.203902e-10,Endesa,Granada,AlmuÃ±Ã©car,ALMUNECA,...,Congested,8,0,0,0.0,2.104539,Near node,False,36.757602,-3.670511


## Current feasibility picture

At this stage, the notebook is starting from a candidate set that already combines mobility need with a first grid-feasibility screen. Before prioritizing sites, we first check how the current candidates are distributed by grid status, route, distributor, and matching distance.

In [2]:
# Quick diagnostic summary

print("Grid-status counts")
display(
    candidates["grid_status"]
    .value_counts(dropna=False)
    .rename_axis("grid_status")
    .reset_index(name="n_candidates")
)

print("\nGrid-match flag counts")
display(
    candidates["grid_match_flag"]
    .value_counts(dropna=False)
    .rename_axis("grid_match_flag")
    .reset_index(name="n_candidates")
)

print("\nDistributor counts")
display(
    candidates["distributor_network"]
    .value_counts(dropna=False)
    .rename_axis("distributor_network")
    .reset_index(name="n_candidates")
)

print("\nTop routes")
display(
    candidates["carretera"]
    .value_counts()
    .rename_axis("carretera")
    .reset_index(name="n_candidates")
    .head(15)
)

print("\nDistance summary (km)")
display(candidates["dist_to_grid_km"].describe())

print("\nAvailable-capacity values")
display(
    candidates["available_capacity_mw"]
    .value_counts(dropna=False)
    .rename_axis("available_capacity_mw")
    .reset_index(name="n_candidates")
    .sort_values("available_capacity_mw", na_position="last")
)

Grid-status counts


,grid_status,n_candidates
0,Congested,44
1,Sufficient,9
2,Moderate,4



Grid-match flag counts


,grid_match_flag,n_candidates
0,Medium distance to node,29
1,Far from nearest node,23
2,Near node,5



Distributor counts


...

,carretera,n_candidates
0,N-502,10
1,N-420,8
2,A-66,7
3,N-120,6
4,N-630,5
5,N-322,5
6,A-7,4
7,N-232,4
8,N-234,4
9,N-432,4



Distance summary (km)


,dist_to_grid_km
count,57.000000
mean,14.339734
std,7.624277
min,2.104539
25%,8.295009
50%,13.579278
75%,18.535076
max,38.313418



Available-capacity values


,available_capacity_mw,n_candidates
0,0.00,41
10,0.91,1
4,4.00,2
6,6.00,1
2,18.67,2
11,19.50,1
1,20.30,3
5,21.27,1
3,27.47,2
7,32.10,1


## Prioritize the candidate set

The candidate layer is dominated by `Congested` sites, but the congestion is not randomly distributed. Several corridors appear repeatedly, which suggests that some roads carry stronger unresolved need than others.

So the next step is to rank candidates using three signals already available in the data:
- grid feasibility,
- distance to the nearest node,
- and repeated corridor presence.

This does not replace the later 2027 demand layer. It is only a first prioritization rule to separate likely deployment sites from likely friction points.

In [3]:
# Corridor pressure + first prioritization rule

# 1) How many candidate points fall on each road?
route_counts = (
    candidates["carretera"]
    .value_counts()
    .rename("route_candidate_count")
    .reset_index()
    .rename(columns={"index": "carretera"})
)

candidates = candidates.merge(route_counts, on="carretera", how="left")

# 2) Route-need class from repeated corridor presence
def classify_route_need(n):
    if n >= 7:
        return "High"
    elif n >= 5:
        return "Medium"
    else:
        return "Low"

candidates["route_need"] = candidates["route_candidate_count"].apply(classify_route_need)

# 3) Distance class
def classify_distance(d):
    if pd.isna(d):
        return "Far"
    elif d <= 5:
        return "Near"
    elif d <= 12:
        return "Medium"
    else:
        return "Far"

candidates["distance_class"] = candidates["dist_to_grid_km"].apply(classify_distance)

# 4) Simple scoring
grid_score_map = {"Sufficient": 3, "Moderate": 2, "Congested": 1}
route_score_map = {"High": 3, "Medium": 2, "Low": 1}
distance_score_map = {"Near": 3, "Medium": 2, "Far": 1}

candidates["grid_score"] = candidates["grid_status"].map(grid_score_map)
candidates["route_score"] = candidates["route_need"].map(route_score_map)
candidates["distance_score"] = candidates["distance_class"].map(distance_score_map)

candidates["priority_score"] = (
    candidates["grid_score"] +
    candidates["route_score"] +
    candidates["distance_score"]
)

# 5) Decision bucket
def assign_decision_bucket(row):
    if row["grid_status"] == "Sufficient":
        return "Deployment-ready"
    elif row["route_need"] in ["High", "Medium"] and row["distance_class"] in ["Near", "Medium"]:
        return "Strategic friction point"
    else:
        return "Lower-priority friction"

candidates["decision_bucket"] = candidates.apply(assign_decision_bucket, axis=1)

print("Decision buckets")
display(
    candidates["decision_bucket"]
    .value_counts(dropna=False)
    .rename_axis("decision_bucket")
    .reset_index(name="n_candidates")
)

print("\nBy route and decision bucket")
display(
    candidates.groupby(["carretera", "decision_bucket"])
    .size()
    .reset_index(name="n_candidates")
    .sort_values(["n_candidates", "carretera"], ascending=[False, True])
    .head(20)
)

print("\nTop candidates by priority score")
display(
    candidates.sort_values(
        ["priority_score", "grid_score", "route_score", "distance_score"],
        ascending=False
    )[
        [
            "candidate_id_n4",
            "carretera",
            "provincia",
            "distributor_network",
            "substation_name",
            "grid_status",
            "dist_to_grid_km",
            "route_candidate_count",
            "route_need",
            "distance_class",
            "priority_score",
            "decision_bucket"
        ]
    ].head(20)
)

Decision buckets


,decision_bucket,n_candidates
0,Lower-priority friction,37
1,Strategic friction point,11
2,Deployment-ready,9



By route and decision bucket


,carretera,decision_bucket,n_candidates
17,N-502,Lower-priority friction,7
13,N-420,Lower-priority friction,5
1,A-66,Lower-priority friction,4
8,N-232,Lower-priority friction,4
9,N-234,Lower-priority friction,4
4,A-7,Lower-priority friction,3
7,N-120,Strategic friction point,3
10,N-322,Lower-priority friction,3
12,N-420,Deployment-ready,3
15,N-432,Lower-priority friction,3



Top candidates by priority score


,candidate_id_n4,carretera,provincia,distributor_network,substation_name,grid_status,dist_to_grid_km,route_candidate_count,route_need,distance_class,priority_score,decision_bucket
32,33,N-420,Cuenca,Endesa,CIUDADELA,Sufficient,4.852041,8,High,Near,9,Deployment-ready
33,34,N-420,Cuenca,Endesa,CIUDADELA,Sufficient,10.336364,8,High,Medium,8,Deployment-ready
6,7,A-66,Zamora,i-DE,ZAMORA,Sufficient,21.846832,7,High,Far,7,Deployment-ready
34,35,N-420,Cuenca,Endesa,CIUDADELA,Sufficient,13.579278,8,High,Far,7,Deployment-ready
50,51,N-502,Ciudad Real,Endesa,S.JORGE,Sufficient,24.019132,10,High,Far,7,Deployment-ready
51,52,N-502,Ciudad Real,Endesa,S.JORGE,Sufficient,15.525812,10,High,Far,7,Deployment-ready
12,13,N-120,Ourense,i-DE,S.MARTIN,Sufficient,26.189298,6,Medium,Far,6,Deployment-ready
10,11,A-7,Alicante/Alacant,i-DE,S.VICENT,Sufficient,8.549634,4,Low,Medium,6,Deployment-ready
0,1,A-66,Cáceres,i-DE,PANTANO GARGUERA,Moderate,23.909121,7,High,Far,6,Lower-priority friction
30,31,N-420,Cuenca,Endesa,MILLOR,Moderate,16.522747,8,High,Far,6,Lower-priority friction


## Reduce the candidate universe

At this stage, the goal is no longer to keep every candidate. The objective is to retain the strongest immediate sites and a limited set of strategically important friction points, while removing the weaker tail of the candidate universe.

In [4]:
# Build a preliminary kept set

work = candidates.copy()

# 1) Avoid duplicated road/substation combinations
work = work.sort_values(
    ["priority_score", "available_capacity_mw", "dist_to_grid_km"],
    ascending=[False, False, True]
).drop_duplicates(subset=["carretera", "substation_name"], keep="first")

# 2) Keep all deployment-ready sites
deployment_ready = work[work["decision_bucket"] == "Deployment-ready"].copy()

# 3) Keep only the strongest strategic friction points per route
strategic = work[work["decision_bucket"] == "Strategic friction point"].copy()

def keep_limit(route_need):
    if route_need == "High":
        return 2
    elif route_need == "Medium":
        return 1
    else:
        return 0

strategic["keep_rank_within_route"] = (
    strategic.sort_values(
        ["carretera", "priority_score", "dist_to_grid_km", "available_capacity_mw"],
        ascending=[True, False, True, False]
    )
    .groupby("carretera")
    .cumcount() + 1
)

strategic["keep_limit"] = strategic["route_need"].apply(keep_limit)

strategic_kept = strategic[strategic["keep_rank_within_route"] <= strategic["keep_limit"]].copy()

# 4) Final preliminary proposal set
proposal = pd.concat([deployment_ready, strategic_kept], ignore_index=True)

proposal = proposal.sort_values(
    ["decision_bucket", "priority_score", "route_need", "dist_to_grid_km"],
    ascending=[True, False, True, True]
).reset_index(drop=True)

print("Preliminary proposal size:", len(proposal))

print("\nProposal by decision bucket")
display(
    proposal["decision_bucket"]
    .value_counts(dropna=False)
    .rename_axis("decision_bucket")
    .reset_index(name="n_sites")
)

print("\nProposal by route")
display(
    proposal["carretera"]
    .value_counts()
    .rename_axis("carretera")
    .reset_index(name="n_sites")
)

print("\nProposal preview")
display(
    proposal[
        [
            "candidate_id_n4",
            "carretera",
            "provincia",
            "distributor_network",
            "substation_name",
            "grid_status",
            "route_need",
            "distance_class",
            "priority_score",
            "decision_bucket"
        ]
    ]
)

Preliminary proposal size: 12

Proposal by decision bucket


,decision_bucket,n_sites
0,Deployment-ready,6
1,Strategic friction point,6



Proposal by route


,carretera,n_sites
0,A-66,3
1,N-502,2
2,N-120,2
3,N-420,1
4,A-7,1
5,N-432,1
6,N-630,1
7,N-322,1



Proposal preview


,candidate_id_n4,carretera,provincia,distributor_network,substation_name,grid_status,route_need,distance_class,priority_score,decision_bucket
0,33,N-420,Cuenca,Endesa,CIUDADELA,Sufficient,High,Near,9,Deployment-ready
1,52,N-502,Ciudad Real,Endesa,S.JORGE,Sufficient,High,Far,7,Deployment-ready
2,7,A-66,Zamora,i-DE,ZAMORA,Sufficient,High,Far,7,Deployment-ready
3,11,A-7,Alicante/Alacant,i-DE,S.VICENT,Sufficient,Low,Medium,6,Deployment-ready
4,13,N-120,Ourense,i-DE,S.MARTIN,Sufficient,Medium,Far,6,Deployment-ready
5,42,N-432,Córdoba,Endesa,LANCHA,Sufficient,Low,Far,5,Deployment-ready
6,5,A-66,Zamora,i-DE,VENTOSA,Congested,High,Medium,6,Strategic friction point
7,49,N-502,Ávila,i-DE,SAN MARTIN,Congested,High,Medium,6,Strategic friction point
8,2,A-66,León,i-DE,VILLAMAÃAN,Congested,High,Medium,6,Strategic friction point
9,55,N-630,Sevilla,Endesa,CALA,Congested,Medium,Near,6,Strategic friction point


## Assign a preliminary charger count

At this stage, charger counts are assigned as a first planning rule rather than a final demand model. The idea is simple: stronger and more feasible sites receive larger proposed installations, while strategically important but grid-constrained sites receive smaller proposed counts.

This remains preliminary and should later be refined using the 2027 demand layer.

In [5]:
# Preliminary charger-count rule

def assign_n_chargers(row):
    # strongest immediate sites
    if row["decision_bucket"] == "Deployment-ready":
        if row["route_need"] == "High":
            return 4
        else:
            return 3

    # important but grid-constrained sites
    if row["decision_bucket"] == "Strategic friction point":
        if row["route_need"] == "High" and row["distance_class"] == "Near":
            return 3
        else:
            return 2

    return 1

proposal["n_chargers_proposed"] = proposal.apply(assign_n_chargers, axis=1)
proposal["estimated_demand_kw"] = proposal["n_chargers_proposed"] * 150

print("Charger-count summary")
display(
    proposal["n_chargers_proposed"]
    .value_counts()
    .rename_axis("n_chargers_proposed")
    .reset_index(name="n_sites")
    .sort_values("n_chargers_proposed")
)

print("\nProposal with charger counts")
display(
    proposal[
        [
            "candidate_id_n4",
            "carretera",
            "provincia",
            "grid_status",
            "route_need",
            "distance_class",
            "decision_bucket",
            "n_chargers_proposed",
            "estimated_demand_kw"
        ]
    ]
)

Charger-count summary


,n_chargers_proposed,n_sites
0,2,6
2,3,3
1,4,3



Proposal with charger counts


,candidate_id_n4,carretera,provincia,grid_status,route_need,distance_class,decision_bucket,n_chargers_proposed,estimated_demand_kw
0,33,N-420,Cuenca,Sufficient,High,Near,Deployment-ready,4,600
1,52,N-502,Ciudad Real,Sufficient,High,Far,Deployment-ready,4,600
2,7,A-66,Zamora,Sufficient,High,Far,Deployment-ready,4,600
3,11,A-7,Alicante/Alacant,Sufficient,Low,Medium,Deployment-ready,3,450
4,13,N-120,Ourense,Sufficient,Medium,Far,Deployment-ready,3,450
5,42,N-432,Córdoba,Sufficient,Low,Far,Deployment-ready,3,450
6,5,A-66,Zamora,Congested,High,Medium,Strategic friction point,2,300
7,49,N-502,Ávila,Congested,High,Medium,Strategic friction point,2,300
8,2,A-66,León,Congested,High,Medium,Strategic friction point,2,300
9,55,N-630,Sevilla,Congested,Medium,Near,Strategic friction point,2,300


The charger-count rule preserves a difference between immediate deployment sites and friction points. In this version, the proposal does not assume that all selected sites should be built at the same scale: stronger sites receive larger installations, while constrained sites are treated more cautiously.

## Build File 2 and File 3

The proposal set is now small enough to be translated into the datathon output structure. File 2 keeps all selected proposed sites, while File 3 keeps only the sites that still behave like friction points from the grid side.

In [6]:
# File 2: Proposed charging locations

proposal = proposal.reset_index(drop=True).copy()
proposal["location_id"] = [f"IBE_{i:03d}" for i in range(1, len(proposal) + 1)]

file2 = proposal[
    [
        "location_id",
        "latitude",
        "longitude",
        "carretera",
        "n_chargers_proposed",
        "grid_status"
    ]
].rename(columns={"carretera": "route_segment"}).copy()

print("File 2 preview")
display(file2)

print("File 2 shape:", file2.shape)
print("File 2 columns:", list(file2.columns))

File 2 preview


,location_id,latitude,longitude,route_segment,n_chargers_proposed,grid_status
0,IBE_001,40.013136,-2.199991,N-420,4,Sufficient
1,IBE_002,38.867801,-4.794636,N-502,4,Sufficient
2,IBE_003,41.339153,-5.722461,A-66,4,Sufficient
3,IBE_004,38.465306,-0.583200,A-7,3,Sufficient
4,IBE_005,42.470048,-6.873236,N-120,3,Sufficient
5,IBE_006,37.791949,-4.703515,N-432,3,Sufficient
6,IBE_007,41.944338,-5.634713,A-66,2,Congested
7,IBE_008,40.326241,-5.012126,N-502,2,Congested
8,IBE_009,42.385657,-5.595729,A-66,2,Congested
9,IBE_010,37.677171,-6.184608,N-630,2,Congested


File 2 shape: (12, 6)
File 2 columns: ['location_id', 'latitude', 'longitude', 'route_segment', 'n_chargers_proposed', 'grid_status']


In [7]:
# File 3: Friction points

file3 = proposal[proposal["grid_status"].isin(["Moderate", "Congested"])].copy()
file3 = file3.reset_index(drop=True)

file3["bottleneck_id"] = [f"FRIC_{i:03d}" for i in range(1, len(file3) + 1)]

file3 = file3[
    [
        "bottleneck_id",
        "latitude",
        "longitude",
        "carretera",
        "distributor_network",
        "estimated_demand_kw",
        "grid_status"
    ]
].rename(columns={"carretera": "route_segment"}).copy()

print("File 3 preview")
display(file3)

print("File 3 shape:", file3.shape)
print("File 3 columns:", list(file3.columns))

File 3 preview


,bottleneck_id,latitude,longitude,route_segment,distributor_network,estimated_demand_kw,grid_status
0,FRIC_001,41.944338,-5.634713,A-66,i-DE,300,Congested
1,FRIC_002,40.326241,-5.012126,N-502,i-DE,300,Congested
2,FRIC_003,42.385657,-5.595729,A-66,i-DE,300,Congested
3,FRIC_004,37.677171,-6.184608,N-630,Endesa,300,Congested
4,FRIC_005,42.363649,-3.364046,N-120,i-DE,300,Congested
5,FRIC_006,39.096867,-1.800315,N-322,i-DE,300,Congested


File 3 shape: (6, 7)
File 3 columns: ['bottleneck_id', 'latitude', 'longitude', 'route_segment', 'distributor_network', 'estimated_demand_kw', 'grid_status']


At this point, the proposal has been converted into the two main operational tables required by the datathon. File 2 represents the selected deployment proposal, while File 3 isolates the subset of locations where mobility need remains visible but grid conditions still appear constrained.

In [8]:
# Save File 2 and File 3

file2.to_csv("/content/File 2.csv", index=False)
file3.to_csv("/content/File 3.csv", index=False)

print("Saved:")
print("- /content/File 2.csv")
print("- /content/File 3.csv")

Saved:
- /content/File 2.csv
- /content/File 3.csv


## Build File 1

File 1 is the summary scorecard for the final proposal. It combines the size of the proposed network, the baseline charging network, the number of friction points, and the projected EV fleet for 2027.

The 2027 EV total must be taken directly from the mandatory datos.gob.es forecasting notebook, so this value should be filled using the official notebook output.

In [9]:
# File 1: Global Network KPIs

# Fill these two values from your earlier notebooks / official forecasting notebook
total_existing_stations_baseline = 4315
total_ev_projected_2027 = 276831

file1 = pd.DataFrame([{
    "total_proposed_stations": len(file2),
    "total_existing_stations_baseline": total_existing_stations_baseline,
    "total_friction_points": len(file3),
    "total_ev_projected_2027": total_ev_projected_2027
}])

print("File 1 preview")
display(file1)

print("File 1 shape:", file1.shape)
print("File 1 columns:", list(file1.columns))

File 1 preview


,total_proposed_stations,total_existing_stations_baseline,total_friction_points,total_ev_projected_2027
0,12,4315,6,276831


File 1 shape: (1, 4)
File 1 columns: ['total_proposed_stations', 'total_existing_stations_baseline', 'total_friction_points', 'total_ev_projected_2027']


In [11]:
file1.to_csv("/content/File 1.csv", index=False)

print("Saved:")
print("- /content/File 1.csv")
print("- /content/File 2.csv")
print("- /content/File 3.csv")

Saved:
- /content/File 1.csv
- /content/File 2.csv
- /content/File 3.csv


At this point, the notebook has produced the three required datathon output tables. The proposal can now be read both as an operational deployment set (File 2) and as a strategic diagnosis of remaining bottlenecks (File 3), summarized in File 1.

The 2027 demand layer is currently incorporated at the scorecard level through the official projected EV total, while site prioritization and charger sizing remain preliminary planning rules.